# Multi-Node Profiling with Nsight Systems
---

Profiling applications across multiple nodes requires careful setup and coordination. In this notebook, we take the [DDP code](../source_code/slurm_ddp.py) and profile it on multiple nodes by submitting a batch script to *Slurm* using `sbatch`. 


**Now, inspect the [script](../source_code/slurm/ddp_multinode.slurm) and follow the steps below to profile the application across multiple nodes:**

- **Step 1**: Open a terminal from **File > New > Terminal**.
   <img src="images/menu.png" width="50%">

- **Step 2**: Navigate to the project workspace folder. It should be in a format of **{username}@slogin001:/lustre/fs01/bootcamps/bootcamp_workspaces/{username}/workspace$**
    <img src="images/terminal.png" width="50%">

- **Step 3**: Use the **sbatch** command to submit the Slurm script by pasting the following command on the terminal and pressing the *Enter* key:

 `sbatch workspace/source_code/slurm/ddp_multinode.slurm`

- **Step 4**: Once done, please run the **squeue --me** command to see if the job was submitted successfully. This will show the list of jobs you submitted.

````text

 JOBID PARTITION   NAME       USER ST         TIME    TIME_LEFT  CPUS MIN_MEM  NODE NODELIST(REASON)
 14695 primary   ddpslurm     tade  R         0:05     11:59:55   512    0      2   gpu[001-002]
````
- On successful execution, you can check for the output progress (`ddpslurm.outputxxx`) in the workspace directory. In case you could not find the output in the `workspace` folder, please check for warnings and errors in the error file `ddpslurm.errorxxx.`, where `xxx` is the JOBID.

Likely output from the `ddpslurm.outputxxx`(<NAME.outputJOBID>).

```python
...
Local rank  1
Local rank  0
Local rank  1
Local rank  2
Local rank  0
Local rank  3
NCCL version 2.21.5+cuda12.4
Local rank  2
Local rank  3
Local Rank: 3, Epoch: 0, Training ...
Local Rank: 0, Epoch: 0, Training ...
Local Rank: 2, Epoch: 0, Training ...
Local Rank: 1, Epoch: 0, Training ...
Local Rank: 1, Epoch: 0, Training ...
Local Rank: 2, Epoch: 0, Training ...
Local Rank: 3, Epoch: 0, Training ...
Local Rank: 0, Epoch: 0, Training ...
---------------------------------------------------------------------------
Epoch: 0, Accuracy: 0.0
---------------------------------------------------------------------------
---------------------------------------------------------------------------
Epoch: 0, Accuracy: 0.0
---------------------------------------------------------------------------
Local Rank: 3, Epoch: 1, Training ...
Local Rank: 2, Epoch: 1, Training ...
Local Rank: 2, Epoch: 1, Training ...
...
```

Once you submit the job, it will take a few minutes, based on how busy the system is, to generate the profiler output files. The given sbatch script will run on two nodes (4 GPUs each) and generate 8 Nsight Systems profiling reports in the `reports/` directory. You can simply navigate to the folder and download the reports, or run the cell below and download the compressed file.

In [ ]:
!cd ../reports && tar -cvf reports.tar *.nsys-rep

After executing the above cell, you should be able to download and save the tar file by holding down Shift and right-clicking [Here](../report/reports.tar) then choosing save Link As. Once done, open the Nsight Systems and follow the below steps to use multi-report feature.

### Viewing Multiple Reports in the Same Timeline
You can open several reports in a single timeline. This could be done using one of these methods:

- **File > Open…** in the main menu, and select several report files.

- **File > New multi-report view** in the main menu, add report files that you want to open in the Multi-report view, and click the “Apply” button.


<img src="images/new-multi-report-view.png">

Multi-report view contains simple editor that allows to add/remove some report files and will load them all on a single timeline after applying that set of reports. When reports are loaded, one can use the *View Selector* to open the Multi-report view again, change the set of reports, and click on “Apply” button to reload the timeline with the new set of reports.

<img src="images/multi-report-view.png">
 
The selected set of reports can be saved as a Multi-report view document and could be opened later to load the same set again.

Now, let's have a look at the example profiler report. If you collapse the `gpuxxx` rows, you can see the number of nodes used for profiling as shown in the screenshot below.

<img src="images/p1.png">

If you look into each node, you see all the collected metrics and data, including CPU, NIC, NVTX, and CUDA traces.
<img src="images/p2.png">

Similar to single-GPU and multi-GPU profiling with Nsight Systems, you can expand the `CUDA HW` row to inspect CUDA API calls, NVTX, Kernels, NCCL communication, and memory usage, and see where the bottlenecks are.

<img src="images/p3.png">

For instance, when we hover the mouse over the specific kernel (as shown below), we can see details such as the number of grids, blocks, duration, and theoretical occupancy, among other metrics. In this case, we can see that NCCL automatically selected Tree-based ALLReduce over Ring-based ALLReduce. The optimisation happens automatically, but you can always enforce NCCL to use Ring-based ALLReduce if needed by using `export NCCL_ALGO=Ring`. 
<img src="images/p4.png">

Below is a simplified view of both the Tree and Ring structures as a reference (8 GPUs across 2 nodes).
```bash
#Tree order
              GPU0
             /    \
          GPU1    GPU4
         /   \    /   \
     GPU2  GPU3 GPU5 GPU6
                        \
                        GPU7

#Ring order
Node 0: GPU0 → GPU1 → GPU2 → GPU3
Node 1: GPU4 → GPU5 → GPU6 → GPU7
        ↑                           ↓
        ← ← ← ← ← ← ← ← ← ← ← ← ← ←
GPU0 → GPU1 → GPU2 → GPU3 → GPU4 → GPU5 → GPU6 → GPU7 → GPU0
```

By adding `export NCCL_DEBUG=INFO` to the `sbatch` script, you can enable logs that show the structure, tree levels, and information about the backend transport (P2P, SHM, IB). The logs will be written into `ddpslurm.outputxxx` file. Below are some examples illustrating the kinda of output you see from the logs.

1. Ring Construction
    `Ring 00 : 0 -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 0`
    This tells you the logical ring order NCCL built for collective communication.
    - Each number is a GPU’s global rank
    - The ring loops back from last to first
    - Multiple rings may be created (e.g., Ring 00, Ring 01, …) for parallelism
2. Transport Types
``` bash
   Channel 00 : 0[0] -> 1[0] via P2P/IPC
   Channel 01 : 4[1] -> 5[1] via NET/IB
```
    This tells you how NCCl connects each GPU pair.
    - P2P = GPU-to-GPU (NVLink or PCIe)
    - SHM = shared memory (within node)
    - NET/IB = inter-node (via InfiniBand or Ethernet)
    - Channel XX = a communication channel (NCCL uses many in parallel)
4. Algorithm Selection
``` bash
   AllReduce Ring
   AllReduce Tree
```
    This tells you which algorithm NCCl used for each collective operation.

When put together, you might see the following for the example DDP code. The logs help you better understand how things work under the hood and make it easier to interpret the profiler report.

Each “Ring” line describes the communication path per ring and per channel: `NCCL INFO Ring 00 : 5 -> 0 -> 3`
This means in Ring 0, rank 5 sends to 0, which sends to 3. This is the communication path for channel 0 on this rank. Since this is printed per-rank, you can piece together full rings across ranks. For this example, there are 16 channels, each with its own ring. So we will see 16 lines as `Ring 00 – Ring 15`.

In this example code, the trees are used in NCCL’s tree-based collectives, such as reduce_scatter and broadcast.

Example log would be `Tree 0 : -1 -> 0 -> 1/4/-1`, where
``` bash
    -1 -> 0 = root is 0 (receives from no one, indicated by -1)
	0 -> 1/4 = 0 sends to ranks 1 and 4
	-1 at the end indicates no third child
```
Each tree line defines a directed communication pattern, often duplicated for upward and downward paths. You might see `Tree 0 to Tree 7`, then `Tree 4 : 4 -> 0 -> 1/-1/-1`, which is from the peer node. It indicates bidirectional trees for more balanced communication.

In the logs, you will also have data path divisions: `Channel 00/16 : 0 3 2 1 4 7 6 5`, which shows the ring order across all ranks (local + remote). In other words, in channel 0, data passes in the order of rank 0 → 3 → 2 → 1 → 4 → 7 → 6 → .

There will be information on affinity in the log too. For instance, `Setting affinity for GPU 0 to ff0000,00000000,...` sets CPU affinity for GPU 0 and `NET/0 GPU/0 GPU/3 GPU/2 GPU/1 NET/0`, indicates that NCCL sees this as a network-topology-aware ring, starting and ending at the NIC (NET/0). This is good for scaling across nodes.

You will also have information on the transport type `Channel 00/0 : 5[1] -> 0[0] [receive] via NET/Socket/0`, which tells you that on channel 0, rank 0 on node 0 receives from rank 5 on node 1, over the network. The [1] and [0] indicate the ranks and their local rank on the node.

## Advanced Analysis
Below are some of the metrics one must examine when looking at the profiler report:

<b>Key Metrics to Examine</b>
1. Communication Patterns
    - Look for regular spikes in network activity that correspond to gradient synchronization
    - These typically appear after each backward pass
2. Bandwidth Utilization
    - Check if your network is saturated (approaching theoretical bandwidth limits)
    - Identify periods of low utilization that might indicate computational bottlenecks
3. Message Sizes
    - Large messages indicate all-reduce operations for gradient synchronization
    - Small, frequent messages may indicate inefficient communication patterns
      
<b>Common Patterns</b>
1. Gradient Synchronization Barriers: Regular spikes that align with iteration boundaries
2. Wait Times: Periods where some nodes are idle while waiting for others
3. Load Imbalance: Uneven communication patterns across nodes

<b>Tips for Optimization</b>
1. Compare network activity with GPU utilization to identify if your workload is network-bound
2. Look for overlapping of computation and communication (ideal pattern)
3. Check for serialization points where all nodes must synchronize

Make sure to look for matching NVTX ranges across nodes and identify nodes that take longer to complete iterations. It is important to check whether communication occurs during computation and whether there is workload balance across the nodes. 

By following these steps, you'll get comprehensive insights into your multi-node DDP training performance, helping you identify and resolve bottlenecks for optimal scaling.



<div style="text-align:center; color:#FF0000;"> <b>Please close the Jupyter Notebook and navigate to your local terminal. Run the launch script command again to switch to the Transformer Engine Container before moving further to the next notebook.<br/> </div></center>

## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook](transEng.ipynb) </b> </div></center>

## Nsight Systems Recipe [Optional]

You can follow the steps below to generate stats by using the analysis recipes available on Nsight Systems. The output will be saved in `reports/`. The generated outputs will be in a form of jupyter notebooks and you can open and run the cells as you would normally do.

In [ ]:
# Installation (run once)
!module load nsight-systems/2024.5.1 && which nsys
!python /cm/shared/apps/Nsight-systems/2024.5.1/target-linux-x64/python/packages/nsys_recipe/install.py --current

In [ ]:
# Check available recipes
!nsys recipe --help

In [ ]:
!nsys recipe nccl_sum --output ../reports/nccl_sum_resnet50 --input ../reports/multinode_nvtx_resnet50_opt*

In [ ]:
!nsys recipe nccl_gpu_time_util_map --output ../reports/nccl_gpu_time_util_map_resnet50 --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep

In [ ]:
# NVTX GPU Projection Trace: This recipe provides a trace of NVTX time ranges projected from the CPU onto the GPU.
!nsys recipe nvtx_gpu_proj_trace --output ../reports/nvtx_gpu_proj_trace_resnet50 --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep

In [ ]:
# GPU Projection Summary: This recipe provides a summary of NVTX time ranges projected from the CPU onto the GPU, and their execution times.
!nsys recipe nvtx_gpu_proj_sum --output ../reports/nvtx_gpu_proj_sum_resnet50 --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep

In [ ]:
!nsys recipe gpu_gaps  --output ../reports/gpu_gaps  --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep
!nsys recipe gpu_metric_util_map  --output ../reports/gpu_metric_util_map  --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep 
!nsys recipe nccl_gpu_overlap_trace  --output ../reports/nccl_gpu_overlap_trace  --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep 
!nsys recipe nvtx_sum   --output ../reports/nvtx_sum   --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep 

In [ ]:
# Network Traffic Summary: This recipe provides a summary of the network traffic over NICs and InfiniBand Switches.
!nsys recipe network_sum --output ../reports/network_sum_resnet50 --input ../reports/multinode_nvtx_resnet50_opt*.nsys-rep 

---

## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.